<a href="https://colab.research.google.com/github/weirdoglh/ComBioNetwork/blob/main/book/Lab3/tuningcurve.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How Do Neurons Represent the World?

## From circuits to sensory coding

In the previous labs you built a circuit that generates a rhythm — the pyloric network. You also saw how synapses can change with activity. Now we ask a different question:

> **How does the brain represent specific features of the world — like the orientation of a visual edge, the pitch of a sound, or the location of a touch?**

The answer is not that a single neuron detects each possible input. Instead, neurons are **tuned** — each one responds preferentially to a particular range of stimuli. A population of neurons with different preferences can together represent a stimulus more precisely than any single neuron could alone.

### A concrete example: orientation in visual cortex

Neurons in the primary visual cortex (V1) respond to oriented edges — a bar of light moving at a particular angle. Each V1 neuron has a **preferred orientation**: it fires most strongly when the bar is at that angle, and less strongly as the angle differs. This response profile is called a **tuning curve**.

To perceive the exact angle of an edge, the brain does not look at one neuron — it reads the pattern of activity across a whole population, each neuron contributing its vote. This is called **population coding**.

In this lab you will:
1. Build a tuned neuron by hand and understand what shapes the tuning curve
2. Let plasticity build tuning automatically
3. Ask whether two tuned neurons together can encode enough information to decode a stimulus

> 💡 Write your answers directly into this notebook. You will submit it as a PDF at the end.

Run the cell below to set up the environment.

In [22]:
#@title ▶ Run to initialize lab environment
#@markdown Cell hidden by default
!pip install ipympl ipywidgets stg-net scipy -q

import matplotlib.pyplot as plt
import numpy as np
import scipy.stats
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

from stg_net.neuron import LIF
from stg_net.input import Poisson_generator
from stg_net.conn import Simulator

plt.rcParams.update({
    'axes.labelsize': 12, 'axes.titlesize': 12,
    'font.size': 12, 'legend.fontsize': 10,
    'lines.linewidth': 2., 'xtick.labelsize': 10, 'ytick.labelsize': 10
})
%matplotlib inline

tonic_neuron = {'tau_m':20., 'a':0., 'tau_w':30., 'b':3., 'V_reset':-55.}

# Shared stimulus space
T, dt = 3e2, 0.1
N_in  = 10
xs    = np.linspace(0.05-1e-5, 0.95-1e-5, N_in)   # preferred stimuli of input neurons
ms    = np.linspace(0., 1., 101)                    # stimulus values for plotting tuning curve

def input_rates(mean, std, xs):
    """Gaussian population code: each input neuron fires at a rate proportional
    to how close its preferred stimulus is to the current stimulus (mean, std)."""
    return np.array([max(scipy.stats.norm(mean, std).pdf([x, x-1, 1+x])) for x in xs])

# Pre-compute tuning curve values for all stimuli
rins = np.array([input_rates(m, 0.1, xs) for m in ms])

print('Environment ready.')

Environment ready.


## Part 1: Building a tuned neuron by hand

### How the simulation works

There are **10 input neurons** (blue in the raster), each one responding preferentially to a different point along a stimulus dimension — think of them as 10 V1 neurons each tuned to a different orientation, or 10 auditory neurons tuned to different frequencies.

When you set a stimulus with a particular **location** (mean) and **spread** (std), the input neurons fire at rates determined by how close their preference is to the stimulus — neurons near the stimulus fire fast, neurons far away fire slowly.

There is one **output neuron** (red in the raster). Its response depends on the weights connecting each input neuron to it — the sliders labelled J_o1 through J_o10.

The **tuning curve** (right panel) shows the predicted response of the output neuron to every possible stimulus location along the x-axis. It is computed directly from the weights — no simulation needed. The higher the curve at a given location, the more the output neuron will fire when that stimulus is presented.

**The sliders:**
- **location** — where the stimulus is on the input dimension (0–10)
- **spread** — how broad or narrow the stimulus is
- **J_o1 … J_o10** — the connection weight from each input neuron to the output neuron

---

### Predict — before touching any sliders

*If you set all weights equal, what shape do you expect the tuning curve to have?*

**My prediction:**
```
[Write here]
```

*If you set only the weight for input neuron 5 (J_o5) to a high value and all others to zero, what shape do you predict the tuning curve will have?*

**My prediction:**
```
[Write here]
```

<img src="https://github.com/weirdoglh/ComBioNetwork/blob/main/book/Lab2/sensory-encoding.PNG?raw=true" alt="Encoding network" width="1000"/>

In [30]:
#@title ▶ Run to start manual tuning curve widget
#@markdown Hidden code

# -----------------------
# UI GRID (with highlights)
# -----------------------
wsize = '180px'
grid  = widgets.GridspecLayout(2, N_in + 2)

labels = ['location', 'spread'] + [f'J_o{i}' for i in range(1, N_in+1)]
for i, label in enumerate(labels):

    # Highlight first two columns
    if i == 0:
        bg = '#f39c12'   # orange
    elif i == 1:
        bg = '#27ae60'   # green (updated from purple)
    else:
        bg = '#dddddd'

    grid[0, i] = widgets.HTML(
        f'<div style="text-align:center;font-size:11px;font-weight:bold;background:{bg};padding:4px">{label}</div>',
        layout=widgets.Layout(width=wsize))

# sliders
grid[1, 0] = widgets.FloatSlider(
    value=5, min=0, max=10, step=0.1,
    style={'handle_color': '#f39c12'},
    layout=widgets.Layout(width=wsize, border='2px solid #f39c12'))

grid[1, 1] = widgets.FloatSlider(
    value=1, min=0.1, max=2, step=0.1,
    style={'handle_color': '#27ae60'},
    layout=widgets.Layout(width=wsize, border='2px solid #27ae60'))

for i in range(2, N_in + 2):
    grid[1, i] = widgets.FloatSlider(
        value=0.2, min=0.0, max=0.4, step=0.01,
        layout=widgets.Layout(width=wsize))

con_bars = {'mean': grid[1, 0], 'std': grid[1, 1]}
for i in range(1, N_in + 1):
    con_bars[f'J_o{i}'] = grid[1, i + 1]


# -----------------------
# MAIN UPDATE
# -----------------------
def update_tune(**con_dict):

    cons_arr = np.array(list(con_dict.values()), dtype=float)
    mean_raw, std_raw, ws = cons_arr[0], cons_arr[1], cons_arr[2:]
    mean, std = mean_raw / 10., std_raw / 10.

    # -----------------------
    # SIMULATION
    # -----------------------
    h    = Simulator(dt=dt)
    nrns = [LIF(sim=h) for _ in range(N_in)]
    for nrn in nrns:
        nrn.update(tonic_neuron)

    rdist = input_rates(mean, std, xs)
    rs    = np.array([0.5] + list(rdist)) * 1e2

    noises = [Poisson_generator(sim=h, rate=r*3, start=0, end=int(T/dt)) for r in rs]
    for noise, nrn in zip(noises, nrns):
        nrn.connect(noise, {'ctype':'Static', 'weight':1e0, 'delay':5})

    tps = [['Static']*N_in for _ in range(N_in)]
    con = np.zeros((N_in, N_in))
    con[0] = ws * 5

    dly = np.random.uniform(2., 5., (N_in, N_in))
    specs = [[{'ctype':tps[i][j],'weight':con[i,j],'delay':dly[i,j]}
              for j in range(N_in)] for i in range(N_in)]

    h.connect(nrns, nrns, specs)
    h.run(T)

    # -----------------------
    # ANALYTICAL TUNING
    # -----------------------
    ys = np.array([np.dot(ws, rin) for rin in rins])
    out_rate = len(nrns[0].spikes['times']) / T * 1e3

    # -----------------------
    # FIGURE
    # -----------------------
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # -------- RASTER --------
    ax1 = axes[0]
    for i, nrn in enumerate(nrns):
        color = '#e74c3c' if i == 0 else '#2980b9'
        ax1.eventplot(nrn.spikes['times'], lineoffsets=i,
                      colors=color, linelengths=0.5)
    ax1.set_title(f'Raster (Output = {out_rate:.1f} Hz)', fontsize=11, fontweight='bold')
    ax1.set_xlim([0., T])
    ax1.set_yticks(range(N_in))
    ax1.set_yticklabels(['Output'] + [f'In {i}' for i in range(1, N_in)])

    # -------- TUNING CURVE --------
    ax2 = axes[1]

    ax2.plot(ms * 10, ys, color='#8e44ad', linewidth=2.5)
    ax2.axvline(x=mean_raw, color='#e74c3c', linestyle='--')

    # 🔥 FIXED SCALE
    ax2.set_ylim(0, max(4, np.max(ys) * 1.1))

    ax2.set_title('Tuning curve')
    ax2.set_xlabel('Stimulus location')
    ax2.set_ylabel('Response')

    # mark sensory neurons
    for x in xs * 10:
        ax2.axvline(x=x, color='#2980b9', linestyle=':', alpha=0.3)

    # -------- CONNECTIVITY DIAGRAM --------
    ax3 = axes[2]

    import networkx as nx
    G = nx.DiGraph()

    # nodes
    G.add_node('Stim', color='#f39c12', label='Input')
    for i in range(N_in):
        G.add_node(f'In{i}', color='#2980b9')
    G.add_node('Out', color='#e74c3c')

    # edges: Stim -> Inputs (thickness based on activation)
    stim_edge_weights = input_rates(mean, std, xs)
    for i, sw in enumerate(stim_edge_weights):
        G.add_edge('Stim', f'In{i}', weight=sw, color='#f39c12')

    # edges: Inputs -> Output (thickness based on weights)
    for i, w in enumerate(ws):
        if w > 0:
            G.add_edge(f'In{i}', 'Out', weight=w, color='gray')

    # positions (layered)
    pos = {}
    pos['Stim'] = (-2, -N_in/2)
    for i in range(N_in):
        pos[f'In{i}'] = (0, -i)
    pos['Out'] = (2, -N_in/2)

    # draw nodes
    nx.draw_networkx_nodes(
        G, pos,
        node_color=[G.nodes[n]['color'] for n in G.nodes()],
        node_size=600, ax=ax3)

    nx.draw_networkx_labels(G, pos, font_size=8, ax=ax3)

    # draw edges individually to handle different colors/widths
    for u, v, d in G.edges(data=True):
        nx.draw_networkx_edges(
            G, pos,
            edgelist=[(u, v)],
            width=1 + 4*d['weight'],
            edge_color=d['color'],
            arrows=True,
            ax=ax3,
            alpha=0.6)

    ax3.set_title('Stimulus → Circuit Flow')
    ax3.axis('off')

    plt.tight_layout()
    plt.show()


# -----------------------
# DISPLAY
# -----------------------
display(grid)
display(widgets.interactive_output(update_tune, con_bars))

GridspecLayout(children=(HTML(value='<div style="text-align:center;font-size:11px;font-weight:bold;background:…

Output()

### Explore — shaping the tuning curve

**First: run with all weights equal (all J sliders at 0.2). Observe the tuning curve.**

*What shape is the tuning curve when all weights are equal? What does this mean for the output neuron's selectivity?*

**My answer:**
```
[Write here]
```

---

**Now: set only J_o5 to 0.4 and all others to 0. Observe the tuning curve.**

*What happens to the shape? Where is the peak? Does this match your prediction?*

**My answer:**
```
[Write here]
```

---

**Challenge — try to create each of these tuning curves by adjusting the weights:**

1. **Narrow peak near location 3** — the output neuron responds only to stimuli near location 3
2. **Broad flat response** — the output neuron responds equally to all stimuli
3. **Two peaks** — the output neuron has two preferred locations

Write down approximately which weights you used for each:

| Target shape | Weights that worked |
|-------------|--------------------|
| Narrow peak near 3 | J_o1=, J_o2=, J_o3=, J_o4=, others≈0 |
| Broad flat | |
| Two peaks | |

*What does this tell you about the relationship between connection weights and a neuron's stimulus selectivity?*

**My answer:**
```
[Write here]
```

---

### Real brain connection

In the visual cortex, orientation-tuned neurons have narrow tuning curves — they respond to a roughly 30–40° range of orientations. Each neuron in V1 gets input from many retinal neurons through the lateral geniculate nucleus, and the weights of those connections determine which orientation the neuron prefers. By selecting appropriate weights (through development and experience), the brain creates a neuron tuned to a specific orientation — exactly what you just did by hand.

*The key insight is that tuning is a property of the weights, not of the neuron type. The same LIF neuron can be tuned to any stimulus — or to none at all — purely by changing its input weights. What does this tell you about where "stimulus selectivity" really lives in a neural circuit?*

**My answer:**
```
[Write here]
```

---

### Checkpoint 1

- A neuron's tuning curve describes: ______
- To make an output neuron respond preferentially to stimulus location 5, I would: ______
- A flat tuning curve (equal response to all stimuli) means: ______

## Part 2: Can plasticity create tuning automatically?

You just shaped a tuning curve by hand. But in a real brain, no one sets the weights manually — they emerge through development and experience-dependent plasticity.

The simulation below has **two output neurons (A and B)** and the same 10 input neurons. When **plasticity is ON**, Hebbian learning adjusts the weights as stimuli are presented. When **plasticity is OFF**, the weights are frozen and you can test the neurons' responses.

The stimulus is now described by both a **mean** (location) and a **std** (spread — how broad or narrow the input activation is). The goal is to train neurons A and B to become tuned to different parts of this stimulus space.

**How to train:**
1. Set **plasticity = True**
2. Present stimuli at different **mean** values — move the mean slider across the range
3. After several presentations, set **plasticity = False**
4. Now test: present stimuli at different locations and observe how A and B respond

The plots show:
- **Raster** — activity of all neurons
- **Synaptic weights** — how the weights change over time (only meaningful during plasticity)
- **Tuning curve A / B** — the predicted response of each output neuron across all stimulus values
- **Input patterns** — the stimuli you have presented so far (dots = each presentation)
- **Output responses** — the response of A vs B to each stimulus (useful for decoding)

---

### Predict — before training

*If you train neuron A by mostly presenting stimuli near location 0.2, and neuron B by mostly presenting stimuli near location 0.8 — what do you predict their tuning curves will look like after training?*

**My prediction:**
```
[Write here]
```

*If A and B end up tuned to different locations, do you think it would be possible to figure out which stimulus was presented just by looking at A and B's responses? How?*

**My prediction:**
```
[Write here]
```

In [38]:
#@title ▶ Run to start competitive learning simulation
#@markdown Hidden code
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import scipy.stats
import networkx as nx

# -----------------------
# UI STYLING HELPERS
# -----------------------
def highlight_slider(sl, color):
    sl.layout = widgets.Layout(width='420px', border=f'3px solid {color}', padding='4px')
    return sl

# -----------------------
# MAIN UPDATE FUNCTION
# -----------------------
def update_comp(mean=0.5, std=0.1, Target_Output_Neuron='A', plasticity=False):

    nIdx = out_labels.index(Target_Output_Neuron)

    h    = Simulator(dt=dt)
    nrns = [LIF(sim=h) for _ in range(M + N_in)]
    for nrn in nrns:
        nrn.update(tonic_neuron)

    # -----------------------
    # INPUT
    # -----------------------
    rdist = input_rates(mean, std, xs)
    rs    = np.array([0.5]*M + list(rdist)) * 1e3 / max(np.sum(rdist), 1e-6)

    noises = [Poisson_generator(sim=h, rate=r*3, start=0, end=int(T/dt)) for r in rs]
    for noise, nrn in zip(noises, nrns):
        nrn.connect(noise, {'ctype':'Static', 'weight':1e0, 'delay':5})

    # -----------------------
    # CONNECTIVITY
    # -----------------------
    tps = [['Static']*(M+N_in) for _ in range(M+N_in)]

    if plasticity:
        for i in range(N_in):
            tps[nIdx][i+M] = 'Comp'

    con = np.zeros((M+N_in, M+N_in))
    global ws
    for i in range(M):
        con[i, M:] = ws[i]

    dly  = np.random.uniform(2., 5., (M+N_in, M+N_in))
    specs = [[{'ctype':tps[i][j],'weight':con[i,j],'delay':dly[i,j]}
              for j in range(M+N_in)] for i in range(M+N_in)]

    cons = h.connect(nrns, nrns, specs)
    h.run(T)

    # -----------------------
    # LEARNING UPDATE
    # -----------------------
    if plasticity:
        ws[nIdx] = np.array([cons[nIdx][i+M].weights[-1] for i in range(N_in)])
        global samples, responses
        samples   = []
        responses = []

    # update tuning
    for k in range(M):
        for i in range(len(ms2)):
            for j in range(len(ss)):
                ys2[k, i, j] = np.dot(ws[k], rins2[i, j])

    if not plasticity:
        samples.append([mean, std])
        responses.append([len(nrns[i].spikes['times']) for i in range(M)])

    # -----------------------
    # FIGURE
    # -----------------------
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(3, 3)

    # -------- RASTER --------
    ax1 = fig.add_subplot(gs[0, 0])
    for l, nrn in enumerate(nrns):
        color = '#e74c3c' if l < M else '#2980b9'
        ax1.eventplot(nrn.spikes['times'], lineoffsets=l,
                     colors=color, linelengths=0.4)
    ax1.set_title('Raster (A/B Red, Inputs Blue)')
    ax1.set_yticks(list(range(M+N_in)))
    ax1.set_yticklabels(out_labels + [f'In {i}' for i in range(1, N_in+1)])
    ax1.set_xlim([0., T])

    # -------- CONNECTIVITY GRAPH --------
    ax2 = fig.add_subplot(gs[0, 1])
    G = nx.DiGraph()
    G.add_node('Stim', color='#f39c12', label='Input')
    for i in range(N_in):
        G.add_node(f'In{i}', color='#2980b9')
    for i in range(M):
        G.add_node(out_labels[i], color='#e74c3c')

    pos = {'Stim': (-2, N_in/2)}
    for i in range(N_in): pos[f'In{i}'] = (0, i)
    for i in range(M): pos[out_labels[i]] = (2, i * (N_in/2) + N_in/4)

    nx.draw_networkx_nodes(G, pos, node_color=[G.nodes[n]['color'] for n in G.nodes()], node_size=500, ax=ax2)
    nx.draw_networkx_labels(G, pos, font_size=9, ax=ax2)

    # Stim -> In
    s_weights = input_rates(mean, std, xs)
    for i, sw in enumerate(s_weights):
        nx.draw_networkx_edges(G, pos, edgelist=[('Stim', f'In{i}')], width=1+4*sw, edge_color='#f39c12', alpha=0.5, ax=ax2)
    # In -> Out
    for i in range(M):
        for j in range(N_in):
            if ws[i, j] > 0:
                nx.draw_networkx_edges(G, pos, edgelist=[(f'In{j}', out_labels[i])], width=1+4*ws[i,j], edge_color='gray', alpha=0.5, ax=ax2)

    ax2.set_title("Stimulus → Circuit Flow")
    ax2.axis('off')

    # -------- WEIGHT HEATMAP --------
    ax3 = fig.add_subplot(gs[0, 2])
    im3 = ax3.imshow(ws, aspect='auto', cmap='viridis')
    ax3.set_title("Synaptic Weight Matrix")
    ax3.set_ylabel("Output Neuron (A/B)")
    ax3.set_xlabel("Input Neurons (1-10)")
    plt.colorbar(im3, ax=ax3)

    # -------- TUNING CURVES --------
    for k, lbl in enumerate(out_labels):
        ax = fig.add_subplot(gs[1, k])
        im = ax.imshow(ys2[k], extent=[std_low, std_high, 0., 1.0], origin='lower', aspect='auto', vmin=0, vmax=max(5, np.max(ys2)))
        ax.set_title(f"Tuning Map: Neuron {lbl}")
        ax.set_xlabel("Stim Spread (std)")
        ax.set_ylabel("Stim Location (mean)")
        plt.colorbar(im, ax=ax)

    # -------- STIM/RESPONSE SPACE --------
    ax5 = fig.add_subplot(gs[2, 0])
    if len(samples) > 0:
        s_arr = np.array(samples)
        sc5 = ax5.scatter(s_arr[:,0], s_arr[:,1], c=range(len(samples)), cmap='plasma')
        plt.colorbar(sc5, ax=ax5)
    ax5.set_title("History: Stimuli Presented")
    ax5.set_xlabel("Location")
    ax5.set_ylabel("Spread")

    ax6 = fig.add_subplot(gs[2, 1])
    if len(responses) > 0:
        r_arr = np.array(responses)
        sc6 = ax6.scatter(r_arr[:,0], r_arr[:,1], c=range(len(responses)), cmap='plasma')
        plt.colorbar(sc6, ax=ax6)
    ax6.set_title("Response Space (A vs B)")
    ax6.set_xlabel("Neuron A Spikes")
    ax6.set_ylabel("Neuron B Spikes")

    plt.tight_layout()
    plt.show()

# -----------------------
# WIDGETS
# -----------------------
mean_slider = highlight_slider(widgets.FloatSlider(min=0., max=1., step=0.01, value=0.5, description='Location'), '#f39c12')
std_slider = highlight_slider(widgets.FloatSlider(min=std_low, max=std_high, step=0.01, value=0.1, description='Spread'), '#27ae60')

widgets.interact(update_comp, mean=mean_slider, std=std_slider, Target_Output_Neuron=out_labels, plasticity=[True, False]);

interactive(children=(FloatSlider(value=0.5, description='Location', layout=Layout(border='3px solid #f39c12',…

### Explore — training and testing

**Step 1 — Train neuron A:**
Set plasticity = True, Target = A. Present several stimuli with mean near 0.2. Move the mean slider to 0.2, 0.15, 0.25 a few times. Watch the weight heatmap change.

*What do you notice about the weight heatmap while plasticity is on? Which input neuron weights increase?*

**My answer:**
```
[Write here]
```

---

**Step 2 — Train neuron B:**
Set Target = B (keep plasticity = True). Present several stimuli near mean = 0.8.

*After training B, look at the two tuning curve panels (middle row). Do A and B have different preferred locations?*

**My answer:**
```
[Write here]
```

---

**Step 3 — Test (plasticity OFF):**
Set plasticity = False. Now systematically sweep the mean slider from 0 to 1. Watch the output responses panel (bottom right).

*As you move the stimulus from left (mean ≈ 0) to right (mean ≈ 1), what happens to the A response and the B response? Do they change in the same direction or opposite directions?*

**My answer:**
```
[Write here]
```

*Look at the response space plot (bottom right). Each dot is one stimulus presentation. If different stimuli produce different (A, B) pairs, that means you could decode the stimulus from the neural responses. Do different stimulus locations produce distinguishable response pairs?*

**My answer:**
```
[Write here]
```

---

**Step 4 — Explore what makes decoding work or fail:**

*Try training A and B to the same stimulus location (both near 0.5). Then test. Can you still decode different stimuli from the response space? Why or why not?*

**My answer:**
```
[Write here]
```

<details>
<summary>💡 Hint — if your tuning curves look identical after training</summary>

Competitive plasticity means that when one neuron "wins" (fires more), it updates its weights in the direction of the current stimulus, while the other neuron's weights are less affected. For A and B to develop different tuning curves, you need to train them on different stimuli. Try training A exclusively at low mean values and B exclusively at high mean values.

</details>

---

### The decoding principle

If A responds strongly to stimuli near location 0.2 and B responds strongly to stimuli near location 0.8, then:
- A stimulus near 0.2 → A fires a lot, B fires a little
- A stimulus near 0.8 → B fires a lot, A fires a little
- A stimulus near 0.5 → A and B fire similarly

So the **ratio** of A and B's responses encodes the stimulus location, even though no single neuron knows the full answer. This is population coding.

*With two neurons, you can encode stimulus location (mean). Do you think two neurons would be enough to also independently encode the spread (std)? How many neurons might you need?*

**My answer:**
```
[Write here]
```

---

### Checkpoint 2

- Competitive plasticity shaped the tuning curves by: ______
- For decoding to work, the two neurons' tuning curves must be: ______
- Population coding works because: ______

## Part 3: Putting it all together

You have now seen three principles working together:

1. **Tuning** — connection weights determine what a neuron responds to
2. **Plasticity** — Hebbian/competitive learning can automatically shape those weights
3. **Population coding** — multiple tuned neurons together encode more information than any single neuron

These principles are not limited to sensory systems. They appear wherever the brain needs to represent information — in motor cortex representing movement directions, in hippocampus representing spatial locations, in prefrontal cortex representing abstract categories.

---

### Connecting back to the course

*In the degeneracy lab, you found that the STG circuit can produce the same rhythm with many different weight combinations. In the plasticity lab, you saw that weights change with activity. Now you see that tuning curves are determined by weights. How might these three things fit together — could the brain use plasticity to maintain a stable circuit output (like the pyloric rhythm) even as individual weights drift?*

**My answer:**
```
[Write here]
```

*In this notebook, you explored tuning curves computationally in minutes. In real neuroscience, measuring tuning curves requires recording from neurons while presenting many different stimuli — a process that takes hours per neuron. What aspects of tuning curve research are particularly well-suited to computational modelling?*

**My answer:**
```
[Write here]
```

---

### Final Checkpoint

- A tuning curve describes: ______
- The shape of a neuron's tuning curve is determined by: ______
- Population coding is necessary because: ______
- Competitive plasticity created different tuning curves in A and B by: ______

---

### Bonus — Think deeper

*You trained neurons A and B with different stimuli and they developed different tuning curves. But you had to decide where to train each one. In a real brain, no one makes this decision — neurons develop their tuning through experience with natural stimuli. What properties of natural stimuli (statistics, correlations, frequency of different features) might determine which tuning curves develop through unsupervised plasticity?*

**My answer:**
```
[Write here]
```

*You showed that two neurons with different tuning curves can help decode stimulus location. But real brains use hundreds or thousands of neurons to represent each stimulus dimension. What advantage does a larger population give — beyond just redundancy?*

**My answer:**
```
[Write here]
```